# Steady-state Prediction for Markov Model
This Notebook streamlines the functionality of this module, allowing for interactive parameter modification and prediction.
The main features include:
1. Importing the Markov model analysis library.
2. Configuring the model and input parameters.
3. Executing ligand (T/D) concentration gradient scans.

In [11]:
import numpy as np
from matplotlib import pyplot as plt

# Import local libraries
from include.utils import *
from include.model import MarkovModel
from include.instance_analysis import InstanceAnalysis

In [12]:
# --- Configuration Center ---
# Modify variables that were previously passed via command line here
confs = 'ohca'                # Model variant choice: 'ohc' (3-conf. model) or 'ohca' (4-conf. model)
v_index = 323                 # Variant index
activeConfs = 'c'             # Catalytically active conformations: 'ca' (two) or 'c' (one)
N_param = 2                  # Number of parameter rows to process from the first row
param_file = './example/trained_parameters.sample.csv'  # Path to the input parameter file
work_dir_path = './example/output'          # Output working directory
# --------------------

In [13]:
# 1. Variable Preparation
if confs == "ohc":
    N_var = 18
elif confs == "ohca":
    N_var = 23
else:
    raise ValueError("Invalid confs. Must be 'ohc' or 'ohca'.")

if activeConfs == "ca":
    activeConf_list = [2, 3]
else:
    activeConf_list = [2]

# 2. Initialize Markov Model
model = MarkovModel(confs=confs, v_index=v_index,
                    activeConf=activeConf_list,
                    angle_cat=80, jointConfTrans=False)

# 3. Prepare Work Directory and Analysis Instance
instance = InstanceAnalysis(model, np.zeros(N_var), work_dir_path, gbInteractionEnergy=-2.3026)

In [14]:
# 4. Execute Steady-state Analysis Gradient Scans
print(f"Starting analysis for parameter sets #1~{N_param}...")

for k in range(1, N_param + 1):
    try:
        vec_para, full_rec = readPara(param_file, k)
    except Exception:
        print(f"Reached end of parameter file or error at parameter set: {k}")
        break

    print(f"Current Parameter Set: {k}")
    
    # Update current parameters
    instance.updateParameters(vec_para, k)

    # (A) Core Prediction: T (ATP) concentration gradient scan
    # Fix D (ADP) at high/low concentrations while scanning T
    # This mimics the experimental measurements for ATP titration curves
    instance.scan_conc_gradient(gradient="T",
                                concRange={"T": np.logspace(-9, -1, 17),
                                             "D": [1e-9, 1e-18]})

    # (B) Core Prediction: D (ADP) concentration gradient scan
    # Fix T (ATP) concentration while scanning D
    # This mimics the experimental measurements for ADP titration curves
    instance.scan_conc_gradient(gradient="D",
                                concRange={"D": np.logspace(-9, -1, 17),
                                             "T": [1e-18]})

print("Analysis Completed.")

Starting analysis for parameter sets #1~2...
Current Parameter Set: 1
Current Parameter Set: 2
Analysis Completed.
